In [2]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set pandas display options
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### loads `.env`
#### Setting up the `database connection`

In [3]:
load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

engine = create_engine(pg_url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET search_path TO mart, curated, public;"))

with engine.begin() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-11-14 10:01:53.902907-05:00


`38% (~212796)` fraudulent claims detected out of `558211` total claims
- `IP - 23402` (~11%) fraudulent records<br>
- `OP - 189394` (~89%) fraudulent records

B) `Python — HDBSCAN clustering` of fraudulent claims (IP & OP)

In [7]:
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
import hdbscan

# ---- Load the fraud-only tables
q_ip = "SELECT * FROM mart.fraud_claims_ip"
q_op = "SELECT * FROM mart.fraud_claims_op"
ip = pd.read_sql(q_ip, con=engine)
op = pd.read_sql(q_op, con=engine)

# ---- Feature sets
ip_feats = [
    "reimb_amt",
    "deductible_paid",
    "dx_count", "px_count",
    "los_days",
    "z_ip_reimb", "z_ip_los",
    "dup_exact_flag", "dup_near_count",
    "overcharge_z_flag", "overcharge_iqr_flag",
]
op_feats = [
    "reimb_amt",
    "deductible_paid",
    "dx_count", "px_count",
    "z_op_reimb",
    "dup_exact_flag", "dup_near_count",
    "overcharge_z_flag", "overcharge_iqr_flag",
]

# ---- Minimal cleaning/engineering: log$ and deductible share

def prep_claim_matrix(df, feat_cols, add_ip=False):
    X = df[feat_cols].copy()

    # Replace inf/neg/strange values
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # Log1p of dollars, plus deductible share
    if "reimb_amt" in X.columns:
        X["reimb_amt_log"] = np.log1p(X["reimb_amt"].clip(lower=0))
    if "deductible_paid" in X.columns:
        denom = df["reimb_amt"].replace(0, np.nan)
        # cap share to tame noise
        X["deductible_share"] = (df["deductible_paid"] / denom).clip(0, 5)

    # Drop raw $ columns after deriving
    drop_raw = [c for c in ["reimb_amt", "deductible_paid"] if c in X.columns]
    X = X.drop(columns=drop_raw)

    # Fill NA with medians (robust)
    X = X.fillna(X.median(numeric_only=True))

    # Robust scale
    scaler = RobustScaler()
    Xs = scaler.fit_transform(X)

    # 2D PCA for visualization only
    pca = PCA(n_components=2, random_state=42)
    X2 = pca.fit_transform(Xs)

    return X, Xs, X2


X_ip, Xs_ip, X2_ip = prep_claim_matrix(ip, ip_feats, add_ip=True)
X_op, Xs_op, X2_op = prep_claim_matrix(op, op_feats, add_ip=False)

# ---- HDBSCAN (separate models for IP and OP)
# Start with sensible defaults; tweak min_cluster_size if everything goes to noise
clusterer_ip = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=10)
clusterer_op = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=10)

labels_ip = clusterer_ip.fit_predict(Xs_ip)
labels_op = clusterer_op.fit_predict(Xs_op)

ip["cluster"] = labels_ip
op["cluster"] = labels_op

# ---- Quick cluster size sanity
ip_counts = ip["cluster"].value_counts(dropna=False).rename("n").sort_index()
op_counts = op["cluster"].value_counts(dropna=False).rename("n").sort_index()
print("IP clusters (label:-1 is noise):\n", ip_counts.to_string())
print("\nOP clusters (label:-1 is noise):\n", op_counts.to_string())

# ---- Profiles: medians by cluster (exclude noise=-1 in profiles)
def cluster_profile(df, feat_cols):
    keep = df[df["cluster"] != -1].copy()
    if keep.empty:
        return pd.DataFrame()
    stats = (keep
             .groupby("cluster")[feat_cols + ["reimb_amt", "deductible_paid"]]
             .median(numeric_only=True)
             .reset_index())
    # add counts
    counts = keep["cluster"].value_counts().rename_axis(
        "cluster").reset_index(name="n")
    prof = stats.merge(counts, on="cluster", how="left")
    return prof.sort_values("n", ascending=False)


ip_profile = cluster_profile(
    ip, [c for c in X_ip.columns if c not in ("reimb_amt", "deductible_paid")])
op_profile = cluster_profile(
    op, [c for c in X_op.columns if c not in ("reimb_amt", "deductible_paid")])

# ---- Save outputs
ip[["claimid", "provider", "cluster"]].to_csv(
    "fraud_ip_claims_with_clusters.csv", index=False)
op[["claimid", "provider", "cluster"]].to_csv(
    "fraud_op_claims_with_clusters.csv", index=False)
ip_profile.to_csv("fraud_ip_cluster_profile.csv", index=False)
op_profile.to_csv("fraud_op_cluster_profile.csv", index=False)

# ---- Visualization
def plot_pca_scatter(X2, labels, title):
    plt.figure()
    # Noise first
    mask_noise = (labels == -1)
    plt.scatter(X2[mask_noise, 0], X2[mask_noise, 1],
                s=6, alpha=0.4, label="noise (-1)")
    # Clusters
    for c in np.unique(labels):
        if c == -1:
            continue
        m = (labels == c)
        plt.scatter(X2[m, 0], X2[m, 1], s=10, alpha=0.8, label=f"cluster {c}")
    plt.title(title)
    plt.xlabel("PCA1")
    plt.ylabel("PCA2")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()


plot_pca_scatter(X2_ip, labels_ip,
                 "Fraudulent IP claims — HDBSCAN clusters (PCA view)")
plot_pca_scatter(X2_op, labels_op,
                 "Fraudulent OP claims — HDBSCAN clusters (PCA view)")

c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


IP clusters (label:-1 is noise):
 cluster
-1      7531
 0        40
 1       618
 2       206
 3        47
 4        82
 5       234
 6        33
 7        58
 8        50
 9        49
 10       31
 11       89
 12      104
 13      137
 14      232
 15      181
 16       28
 17       27
 18      539
 19       64
 20       96
 21      106
 22      111
 23      194
 24       54
 25      269
 26       85
 27       61
 28      142
 29      235
 30       78
 31       39
 32       36
 33      115
 34       29
 35      112
 36       25
 37      428
 38      109
 39      143
 40       51
 41      575
 42       98
 43       40
 44      723
 45       99
 46      130
 47       38
 48      938
 49      115
 50      172
 51       42
 52      189
 53      224
 54      113
 55       54
 56      298
 57      150
 58      335
 59       81
 60      292
 61      297
 62       39
 63       80
 64       96
 65      172
 66      128
 67       86
 68       77
 69      161
 70       90
 71      167
 72      

KeyError: "Columns not found: 'deductible_share', 'reimb_amt_log'"